In [1]:
import haystack

print("Haystack version:", haystack.__version__)
print("W7D2 notebook started")

Haystack version: 3.3.0
W7D2 notebook started


In [2]:
from pathlib import Path
from haystack.components.converters import PyPDFToDocument

pdf_folder = Path(
    r"C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks"
)

converter = PyPDFToDocument()
all_documents = []

for pdf_file in sorted(pdf_folder.glob("*.pdf")):
    result = converter.run(sources=[str(pdf_file)])
    documents = result["documents"]
    all_documents.extend(documents)
    print(f"{pdf_file.name}: {len(documents)} document(s)")

print("\nTotal documents:", len(all_documents))

computer_vision_robotics.pdf: 1 document(s)
pid_control.pdf: 1 document(s)
robot_sensors.pdf: 1 document(s)
robotics_reference.pdf: 1 document(s)
ros2_basics.pdf: 1 document(s)

Total documents: 5


In [3]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

document_store.write_documents(all_documents)

print("Documents indexed:", document_store.count_documents())

Documents indexed: 5


In [4]:
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

bm25_retriever = InMemoryBM25Retriever(
    document_store=document_store,
    top_k=3
)

print("BM25 Retriever created")

BM25 Retriever created


In [5]:
from haystack_integrations.components.readers.transformers import TransformersExtractiveReader

reader = TransformersExtractiveReader(
    top_k=2,
    no_answer=False
)

print("Extractive Reader created")

Extractive Reader created


In [6]:
from haystack import Pipeline

bm25_pipeline = Pipeline()

bm25_pipeline.add_component("retriever", bm25_retriever)
bm25_pipeline.add_component("reader", reader)

bm25_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("BM25 → Reader pipeline created")

BM25 → Reader pipeline created


In [7]:
question = "What does LiDAR measure?"

result = bm25_pipeline.run({
    "retriever": {"query": question},
    "reader": {"query": question}
})

answers = result["reader"]["answers"]

for answer in answers:
    print("Answer:", answer.data)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Answer: distance
Answer: distances


In [9]:
questions = [
    "What does LiDAR measure?",
    "What does an IMU measure?",
    "What do wheel encoders measure?",
    "What is the purpose of sensors in robots?",
    "What is SLAM used for?",
    "What is the purpose of a PID controller?",
    "What is ROS 2 used for?",
    "What do cameras provide for robots?",
    "What do ultrasonic sensors estimate?",
    "What do actuators produce?"
]

bm25_answers = []

for i, question in enumerate(questions, 1):
    result = bm25_pipeline.run({
        "retriever": {"query": question},
        "reader": {"query": question}
    })

    answers = result["reader"]["answers"]
    answer = answers[0].data if answers else "No answer"

    bm25_answers.append(answer)

    print(f"Q{i}: {question}")
    print(f"Answer: {answer}")
    print("-" * 50)

Q1: What does LiDAR measure?
Answer: distance
--------------------------------------------------
Q2: What does an IMU measure?
Answer: angular velocity and linear acceleration
--------------------------------------------------
Q3: What do wheel encoders measure?
Answer: wheel rotation
--------------------------------------------------
Q4: What is the purpose of sensors in robots?
Answer: to perceive their surroundings
--------------------------------------------------
Q5: What is SLAM used for?
Answer: allows a robot to build a map while estimating its own position
--------------------------------------------------
Q6: What is the purpose of a PID controller?
Answer: to regulate a system
--------------------------------------------------
Q7: What is ROS 2 used for?
Answer: to build distributed robot applications
--------------------------------------------------
Q8: What do cameras provide for robots?
Answer: image data for computer vision
----------------------------------------------

In [10]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersDocumentEmbedder
)
from haystack.document_stores.types import DuplicatePolicy

document_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

document_embedder.warm_up()

embedded_result = document_embedder.run(
    documents=all_documents
)

document_store.write_documents(
    embedded_result["documents"],
    policy=DuplicatePolicy.OVERWRITE
)

print("Dense document embeddings created")
print("Documents in store:", document_store.count_documents())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Dense document embeddings created
Documents in store: 5


In [11]:
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersTextEmbedder
)

text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

text_embedder.warm_up()

print("Dense text embedder created")

Dense text embedder created


In [12]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=3
)

print("Dense Retriever created")

Dense Retriever created


In [15]:
dense_pipeline = Pipeline()

In [17]:
from haystack import Pipeline
from haystack_integrations.components.embedders.sentence_transformers import (
    SentenceTransformersTextEmbedder
)
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack_integrations.components.readers.transformers import (
    TransformersExtractiveReader
)

fresh_text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
fresh_text_embedder.warm_up()

fresh_dense_retriever = InMemoryEmbeddingRetriever(
    document_store=document_store,
    top_k=3
)

fresh_dense_reader = TransformersExtractiveReader(
    top_k=2,
    no_answer=False
)

dense_pipeline = Pipeline()

dense_pipeline.add_component("text_embedder", fresh_text_embedder)
dense_pipeline.add_component("retriever", fresh_dense_retriever)
dense_pipeline.add_component("reader", fresh_dense_reader)

dense_pipeline.connect(
    "text_embedder.embedding",
    "retriever.query_embedding"
)

dense_pipeline.connect(
    "retriever.documents",
    "reader.documents"
)

print("Dense → Reader pipeline created")

Dense → Reader pipeline created


In [18]:
question = "What does LiDAR measure?"

dense_result = dense_pipeline.run({
    "text_embedder": {"text": question},
    "reader": {"query": question}
})

answers = dense_result["reader"]["answers"]

for answer in answers:
    print("Answer:", answer.data)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer: distance
Answer: distances


In [19]:
dense_answers = []

for i, question in enumerate(questions, 1):
    result = dense_pipeline.run({
        "text_embedder": {"text": question},
        "reader": {"query": question}
    })

    answers = result["reader"]["answers"]
    answer = answers[0].data if answers else "No answer"

    dense_answers.append(answer)

    print(f"Q{i}: {question}")
    print(f"Answer: {answer}")
    print("-" * 50)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q1: What does LiDAR measure?
Answer: distance
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q2: What does an IMU measure?
Answer: angular velocity and linear acceleration
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Q8: What do cameras provide for robots?
Answer: image data for computer vision
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q9: What do ultrasonic sensors estimate?
Answer: distance
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Q10: What do actuators produce?
Answer: movement
--------------------------------------------------


## Evaluation and Comparison

The Haystack RAG pipeline was evaluated using 10 questions across 5 indexed PDF documents.

### BM25 Retriever + Reader
- Questions tested: 10
- Correct answers: 10/10
- Accuracy: 100%

### Dense Retriever + Reader
- Questions tested: 10
- Correct answers: 10/10
- Accuracy: 100%
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`

### Comparison

| Method | Questions | Correct | Accuracy |
|---|---:|---:|---:|
| BM25 + Reader | 10 | 10/10 | 100% |
| Dense + Reader | 10 | 10/10 | 100% |

Both retrieval methods produced correct answers for all 10 questions.

BM25 performs lexical keyword-based retrieval, while Dense Retrieval uses semantic embeddings to find relevant documents.

### Conclusion

Both BM25 and Dense Retrieval achieved 100% accuracy on this evaluation dataset. For this small dataset, both methods performed equally well. Dense Retrieval provides semantic retrieval using embeddings, while BM25 relies on keyword matching.